In [1]:
from importlib.metadata import version
print(version("torch"))

2.8.0+cu129


In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
# 计算参数占用的内存（MB）

def calculate_kv_cache_size(past_key_values):
    """计算past_key_values占用的总内存（单位 MB）
    """

    total_size = 0
    if past_key_values is None:
        return total_size

    # past_key_values: k, v
    for tensor in past_key_values:
        # numel() 返回元素的个数， element_size()返回每个元素的字节数（例如是fp16是 2 个字节）
        total_size += tensor.numel() * tensor.element_size()

    return total_size / (1024 * 1024) # 转换为 MB

多头注意力机制 + KV Cache 的手写实现

In [3]:
class MultiHeadAttenionKVCache(nn.Module):
     """多头注意力机制带KV Cache版本的实现"""
     def __init__(self,dim=512,n_heads=8,use_kv=True):
        super(MultiHeadAttenionKVCache,self).__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim//n_heads
        self.use_kv = use_kv

        # Wq, Wk, Wv, Wo矩阵
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        # 存储历史的 tokens
        self.history_seq = []

     def forward(self,q,k,v,past_key=None,past_value=None,mask=None):
        # Prefill阶段： seq_len > 1
        # Decooding阶段： seq_len = 1
        batch_size, seq_len, _ = q.shape

        if not self.use_kv:
            self.history_seq.append(q)


        #Q,K,V映射
        q = self.q_proj(q)
        if self.use_kv:
            k = self.k_proj(k)
            v = self.v_proj(v)

        #切分多头  shape = [batch,n_heads,seq_len,head_dim]
        q = q.view(batch_size,seq_len,self.n_heads,self.head_dim).transpose(1,2)
        if self.use_kv:
            k = k.view(batch_size,seq_len,self.n_heads,self.head_dim).transpose(1,2)
            v = v.view(batch_size,seq_len,self.n_heads,self.head_dim).transpose(1,2)
        if self.use_kv:
            # KV Cache 在seq_len维度去拼接past_key, past_value
            # history: [batch, n_heads, seq_len, head_dim]
            # new: [batch, n_heads, 1, head_dim]
            # 拼接后：[batch, n_heads, seq_len + 1, head_dim]
            if past_key is not None:
                k = torch.cat([past_key, k], dim=2)
            if past_value is not None:
                v = torch.cat([past_value, v], dim=2)
        else:
            # 标准的多头注意力
            history_seq = torch.cat(self.history_seq, dim=1)
            k = self.k_proj(history_seq)
            v = self.v_proj(history_seq)
            k = k.view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)

        #保存k,v 用于下一次预测
        past_key_values = (k,v)

        #多头注意力得分计算
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        #加 Casual Mask
        if mask is not None:
             # mask是一个bool矩阵，里面的True代表要被mask的内容
            attn_scores = attn_scores.masked_fill(mask, -1e9)

         # 加 Softmax
        attn_weights = F.softmax(attn_scores, dim=-1)

        # [batch, n_heads, seq_len, head_dim]
        attn_output = torch.matmul(attn_weights, v)

        # 拼接多头
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.dim)

        # 输出映射
        output = self.o_proj(attn_output)
        return output, past_key_values







In [9]:
# 超参数
batch = 64
seq_len = 10
dim = 4096
heads = 8
use_kv = True

# 模拟输入，prompt的长度就是seq_len
x = torch.randn(batch, seq_len, dim)

# 构造mask
mask = torch.full((1, 1, seq_len, seq_len), True)
mask = torch.triu(mask, diagonal=1)
# print("mask: ", mask)


mha_cache = MultiHeadAttenionKVCache(dim=dim, n_heads=heads, use_kv=use_kv)

# Prefill阶段
output, (past_k, past_v) = mha_cache(x, x, x, mask=mask)
past_kv_mem = calculate_kv_cache_size((past_k, past_v))
if use_kv:
    print("output:", output.shape, "past KV:", past_k.shape, "kv memory: {} MB".format(past_kv_mem))
else:
    print("output:", output.shape, "past KV:", past_k.shape)


# Decoding阶段
N = 100
x = output
begin = time.time()
for _ in range(N):
    # new_x: [batch, 1, dim]
    new_x = output[:, [-1], :]
    output, (past_k, past_v) = mha_cache(new_x, new_x, new_x, past_key=past_k, past_value=past_v)
    past_kv_mem = calculate_kv_cache_size((past_k, past_v))
    if use_kv:
        print("output:", output.shape, "past KV:", past_k.shape, "kv memory: {} MB".format(past_kv_mem))
    else:
        print("output:", output.shape, "past KV:", past_k.shape)

end = time.time()
print("total cost: ", end - begin)




output: torch.Size([64, 10, 4096]) past KV: torch.Size([64, 8, 10, 512]) kv memory: 20.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 11, 512]) kv memory: 22.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 12, 512]) kv memory: 24.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 13, 512]) kv memory: 26.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 14, 512]) kv memory: 28.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 15, 512]) kv memory: 30.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 16, 512]) kv memory: 32.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 17, 512]) kv memory: 34.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 18, 512]) kv memory: 36.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 19, 512]) kv memory: 38.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 8, 20, 512]) kv memory: 40.0 M

GQA + KVCache的实现

In [15]:
class GQAWithKVCache(nn.Module):
    """分组多头注意力机制带KV Cache版本的实现"""

    def __init__(self, dim=512, n_heads=8, kv_heads=4):
        super().__init__()

        self.dim = dim
        self.n_heads = n_heads
        self.kv_heads = kv_heads
        self.head_dim = dim // n_heads

        # Wq, Wk, Wv, Wo矩阵
        # 真正用于attention的 K/V 数量是 kv_heads
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, kv_heads * self.head_dim, bias=False)

        # 输出映射层
        self.o_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)


    def forward(self, q, k, v, past_key=None, past_value=None, mask=None):
        # Prefill阶段： seq_len > 1
        # Decooding阶段： seq_len = 1
        batch_size, seq_len, _ = q.shape

        # Q, K, V映射
        q = self.q_proj(q) # [B, S, H*D]
        k = self.k_proj(k) # [B, S, G*D]
        v = self.v_proj(v) # [B, S, G*D]

        # 切分多头，shape=[batch, n_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2) # [B, H, S, D]
        k = k.view(batch_size, seq_len, self.kv_heads, self.head_dim).transpose(1, 2) # [B, G, S, D]
        v = v.view(batch_size, seq_len, self.kv_heads, self.head_dim).transpose(1, 2) # [B, G, S, D]

        # KV Cache 在seq_len维度去拼接past_key, past_value
        # history: [batch, n_heads, seq_len, head_dim]
        # new: [batch, n_heads, 1, head_dim]
        # 拼接后：[batch, n_heads, seq_len + 1, head_dim]
        if past_key is not None:
            k = torch.cat([past_key, k], dim=2)
        if past_value is not None:
            v = torch.cat([past_value, v], dim=2)

        # 保存K, V用于下一次预测
        past_key_values = (k, v)

        # 复制操作
        # 比如 H=8, G=4, 那每一个group的 key/value 被复制两次
        # 比如4个group，[1,2,3,4], 复制之后 [1,1,2,2,3,3,4,4], 不是[1,2,3,4,1,2,3,4]
        assert self.n_heads % self.kv_heads == 0
        k = k.repeat_interleave(self.n_heads // self.kv_heads, dim=1) # [B, H, S, D]
        v = v.repeat_interleave(self.n_heads // self.kv_heads, dim=1) # [B, H, S, D]

        # 多头注意力得分计算
        # [batch, n_heads, seq_len, seq_len]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # 加 Casual Mask
        if mask is not None:
            # mask是一个bool矩阵，里面的True代表要被mask的内容
            attn_scores = attn_scores.masked_fill(mask, -1e9)

        # 加 Softmax
        attn_weights = F.softmax(attn_scores, dim=-1)

        # [batch, n_heads, seq_len, head_dim]
        attn_output = torch.matmul(attn_weights, v)

        # 拼接多头
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.dim)

        # 输出映射
        output = self.o_proj(attn_output)
        return output, past_key_values

In [16]:
# 超参数
batch = 64
seq_len = 10
dim = 4096
heads = 8
kv_heads = 4

# 模拟输入，prompt的长度就是seq_len
x = torch.randn(batch, seq_len, dim)

# 构造mask
mask = torch.full((1, 1, seq_len, seq_len), True)
mask = torch.triu(mask, diagonal=1)

gqa_cache = GQAWithKVCache(dim=dim, n_heads=heads, kv_heads=kv_heads)

# Prefill阶段
output, (past_k, past_v) = gqa_cache(x, x, x, mask=mask)
past_kv_mem = calculate_kv_cache_size((past_k, past_v))
print("output:", output.shape, "past KV:", past_k.shape, "kv memory: {} MB".format(past_kv_mem))

# Decoding阶段
N = 100
x = output
begin = time.time()
for _ in range(N):
    # new_x: [batch, 1, dim]
    new_x = output[:, [-1], :]
    output, (past_k, past_v) = gqa_cache(new_x, new_x, new_x, past_key=past_k, past_value=past_v)
    past_kv_mem = calculate_kv_cache_size((past_k, past_v))
    print("output:", output.shape, "past KV:", past_k.shape, "kv memory: {} MB".format(past_kv_mem))

end = time.time()
print("total cost: ", end - begin)

output: torch.Size([64, 10, 4096]) past KV: torch.Size([64, 4, 10, 512]) kv memory: 10.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 11, 512]) kv memory: 11.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 12, 512]) kv memory: 12.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 13, 512]) kv memory: 13.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 14, 512]) kv memory: 14.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 15, 512]) kv memory: 15.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 16, 512]) kv memory: 16.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 17, 512]) kv memory: 17.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 18, 512]) kv memory: 18.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 19, 512]) kv memory: 19.0 MB
output: torch.Size([64, 1, 4096]) past KV: torch.Size([64, 4, 20, 512]) kv memory: 20.0 M